> OLD VERSION v1 - do not use for results. The final Gold is built in 05_gold_all_services, Step 3. Keep this notebook only because it also builds gold_location_lookup, which the search in 05 uses. Do not re-run the scoring cells here: they would overwrite gold_facility_services with v1.


# 04 Gold: trust score per facility x service
**Input:** workspace.default.silver_facilities (team master)  
**Output:** workspace.default.gold_facility_services (one row per facility x service with any evidence)

**Idea.** Every field is a claim. For each service we look for evidence in four independent places in the record (specialties, procedure/capability, equipment, description), add points for each kind of evidence, subtract points for quality flags and implausible claims, and keep the exact text we found so the app can quote it.

| Evidence | Points |
|---|---|
| procedure / capability mentions the service | +30 |
| equipment for the service is listed | +25 |
| description mentions the service | +15 |
| specialty list includes it (weakest: specialty lists are inflated) | +10 |
| a concrete number in the evidence (e.g. "10 dialysis machines", "20-bed") | +10 |
| 24/7 / emergency wording (emergency services only) | +5 |
| scale known (beds or doctors) | +5 |

| Penalty | Points |
|---|---|
| dentist / pharmacy claims a hospital-level service | -40 |
| flag_mixed_identity | -30 |
| clinic with no beds claims a hospital-level service | -15 |
| flag_state_mismatch | -15 |
| flag_duplicate, flag_too_many_specialties, flag_implausible_counts | -10 each |
| flag_coords_far_from_pin | -5 |

**Trust levels** (score capped 0-100). *Hard evidence* = procedure/capability, equipment, description.
- **Verified:** score >= 60, at least 2 kinds of hard evidence, no serious problem (mixed identity, state mismatch, duplicate, implausible facility type)
- **Likely:** score >= 35 and at least 1 kind of hard evidence
- **Claimed only:** everything else (e.g. only listed as a specialty)

Service definitions (cell 2) are medical judgement calls: review and edit them, then Run all.

In [0]:
# Cell 2: service definitions = what counts as evidence for each service (REVIEW THESE: medical judgement)
# specialty: regex matched against the specialties list (camelCase codes like "nephrology", "neonatologyPerinatalMedicine")
# text:      regex matched against procedure + capability items and description sentences
# equipment: regex matched against equipment items
# hospital_level: needs inpatient hospital infrastructure (a dentist / pharmacy / bed-less clinic claiming it is suspicious)
# emergency: time-critical service, so 24/7 / emergency wording adds a little trust
import re
import numpy as np
import pandas as pd

SERVICES = {
    "dialysis": dict(label="Dialysis", hospital_level=False, emergency=False,
        specialty=r"nephrolog",
        text=r"dialysis|ha?emodialysis|renal replacement|\bcapd\b|kidney care unit",
        equipment=r"dialys|dialyz|\bro plant\b"),
    "emergency_c_section": dict(label="Emergency C-section", hospital_level=True, emergency=True,
        specialty=r"obstetric|maternalfetal",
        text=r"c-?section|caesarean|cesarean|\blscs\b",
        equipment=r"operation theat|operating theat|\bot\b|labou?r (room|ward)|delivery (room|suite)"),
    "nicu": dict(label="Newborn intensive care (NICU)", hospital_level=True, emergency=True,
        specialty=r"neonatolog|perinatal",
        text=r"\bnicu\b|neonatal intensive|newborn intensive|\bsncu\b|level (ii|iii|2|3) neonatal",
        equipment=r"incubator|radiant warmer|phototherapy|neonatal ventilator|bubble cpap"),
    "cardiac_surgery": dict(label="Cardiac surgery", hospital_level=True, emergency=False,
        specialty=r"cardiacsurg|cardiothoracic|cardiovascularsurg",
        text=r"cardiac surgery|heart surgery|open[- ]heart|bypass surgery|\bcabg\b|valve (replacement|repair)|cardiothoracic",
        equipment=r"heart[- ]lung machine|cardiopulmonary bypass|cath ?lab"),
    "mri": dict(label="MRI scan", hospital_level=False, emergency=False,
        specialty=r"radiolog",
        text=r"\bmri\b|magnetic resonance",
        equipment=r"\bmri\b|magnetic resonance"),
    "ct_scan": dict(label="CT scan", hospital_level=False, emergency=False,
        specialty=r"radiolog",
        text=r"\bct\b|ct[- ]scan|computed tomography|\bcat scan\b",
        equipment=r"\bct\b|ct[- ]scan|computed tomography|slice ct"),
    "burn_care": dict(label="Burn care", hospital_level=True, emergency=True,
        specialty=r"burn",
        text=r"\bburns? (unit|ward|care|centre|center|icu|treatment|management|surgery)|burn injur|treatment of burns",
        equipment=r"burn"),
    "trauma_emergency": dict(label="Trauma / emergency care", hospital_level=True, emergency=True,
        specialty=r"emergencymedicine|traumasurg|traumatolog",
        text=r"\btrauma (care|centre|center|unit|surgery)|emergency (department|room|ward|services?|care)|\bcasualty\b|accident",
        equipment=r"defibrillator|ambulance|ventilator|crash cart"),
    "icu": dict(label="Intensive care (ICU)", hospital_level=True, emergency=True,
        specialty=r"criticalcare|intensivecare",
        text=r"\bicu\b|intensive care|critical care|\biccu\b|\bmicu\b|\bsicu\b|\bccu\b",
        equipment=r"ventilator|icu bed|multipara(meter)? monitor|central monitoring"),
    "cancer_treatment": dict(label="Cancer treatment (chemo / radiation)", hospital_level=False, emergency=False,
        specialty=r"oncolog",
        text=r"chemotherap|radiotherap|radiation (therapy|oncology)|cancer (treatment|care|surgery|centre|center)|oncology",
        equipment=r"linear accelerator|\blinac\b|brachytherap|cobalt|tomotherapy|pet[- ]ct"),
}
for k, s in SERVICES.items():
    for f in ("specialty", "text", "equipment"):
        s[f + "_re"] = re.compile(s[f], re.I)
print(len(SERVICES), "services:", ", ".join(s["label"] for s in SERVICES.values()))

10 services: Dialysis, Emergency C-section, Newborn intensive care (NICU), Cardiac surgery, MRI scan, CT scan, Burn care, Trauma / emergency care, Intensive care (ICU), Cancer treatment (chemo / radiation)


In [0]:
# Cell 3: find evidence and score every facility x service
silver = spark.table("workspace.default.silver_facilities").toPandas()
as_list = lambda x: [str(i) for i in x] if x is not None and not (isinstance(x, float) and np.isnan(x)) else []

QUANT = re.compile(r"\b\d+\s*-?\s*(bed(ded)?s?|machines?|units?|stations?|slices?|tesla|ventilators?|incubators?|warmers?|cots?|chairs?|theatres?|ots?)\b", re.I)
EMERG = re.compile(r"24\s*[x/×*]\s*7|24 ?hours?|round[- ]the[- ]clock|emergency", re.I)
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
SERIOUS = ["flag_mixed_identity", "flag_state_mismatch", "flag_duplicate"]

def score_one(r, key, s):
    specs, proc = as_list(r["specialties"]), as_list(r["procedure"]) + as_list(r["capability"])
    equip, desc = as_list(r["equipment"]), (r["description"] or "")
    hit_spec  = [x for x in specs if s["specialty_re"].search(x)]
    hit_proc  = [x for x in proc if s["text_re"].search(x)]
    hit_equip = [x for x in equip if s["equipment_re"].search(x) or s["text_re"].search(x)]
    hit_desc  = [x for x in SENT_SPLIT.split(desc) if s["text_re"].search(x)]
    if not (hit_spec or hit_proc or hit_equip or hit_desc):
        return None

    pts, why = 0, []
    if hit_proc:  pts += 30; why.append(f"+30 procedure/capability mentions it ({len(hit_proc)})")
    if hit_equip: pts += 25; why.append(f"+25 equipment listed ({len(hit_equip)})")
    if hit_desc:  pts += 15; why.append("+15 description mentions it")
    if hit_spec:  pts += 10; why.append("+10 listed as a specialty")
    hard_text = hit_proc + hit_equip + hit_desc
    quantified = [x for x in hard_text if QUANT.search(x)]
    if quantified: pts += 10; why.append("+10 concrete numbers in the evidence")
    if s["emergency"] and any(EMERG.search(x) for x in proc + [desc]):
        pts += 5; why.append("+5 24/7 or emergency wording")
    if pd.notna(r["n_beds"]) or pd.notna(r["n_doctors"]):
        pts += 5; why.append("+5 size known (beds/doctors)")

    pen, problems = 0, []
    ftype = r["facility_type"]
    type_implausible = s["hospital_level"] and ftype in ("dentist", "pharmacy")
    if type_implausible: pen += 40; problems.append(f"-40 a {ftype} claiming a hospital-level service")
    if s["hospital_level"] and ftype == "clinic" and pd.isna(r["n_beds"]):
        pen += 15; problems.append("-15 clinic with no beds listed claims a hospital-level service")
    for flag, p, txt in [("flag_mixed_identity", 30, "record may merge two different places"),
                         ("flag_state_mismatch", 15, "address state conflicts with PIN code"),
                         ("flag_duplicate", 10, "duplicate name in the same city"),
                         ("flag_too_many_specialties", 10, "specialty list looks inflated"),
                         ("flag_implausible_counts", 10, "doctor/bed numbers look unrealistic"),
                         ("flag_coords_far_from_pin", 5, "coordinates disagree with PIN code")]:
        if r[flag]:
            pen += p; problems.append(f"-{p} {txt}")

    score = int(max(0, min(100, pts - pen)))
    n_hard = sum(bool(h) for h in (hit_proc, hit_equip, hit_desc))
    serious = type_implausible or any(r[f] for f in SERIOUS)
    if score >= 60 and n_hard >= 2 and not serious:
        level = "Verified"
    elif score >= 35 and n_hard >= 1:
        level = "Likely"
    else:
        level = "Claimed only"

    # evidence to quote: concrete-number items first, then the rest; each labelled with its field
    labelled = ([("procedure/capability", x) for x in hit_proc] + [("equipment", x) for x in hit_equip]
                + [("description", x) for x in hit_desc] + [("specialty", x) for x in hit_spec])
    labelled.sort(key=lambda fx: (not QUANT.search(fx[1]), fx[0] == "specialty"))
    evidence = [f"{f}: {x.strip()[:240]}" for f, x in labelled[:5]]

    return dict(unique_id=r["unique_id"], service=key, service_label=s["label"],
                trust_score=score, trust_level=level, n_hard_evidence=n_hard,
                ev_procedure=bool(hit_proc), ev_equipment=bool(hit_equip),
                ev_description=bool(hit_desc), ev_specialty=bool(hit_spec), ev_quantified=bool(quantified),
                evidence=evidence, score_reasons=why, problems=problems,
                points_evidence=int(pts), points_penalty=int(pen))

rows = []
for r in silver.to_dict("records"):
    for key, s in SERVICES.items():
        out = score_one(r, key, s)
        if out:
            rows.append(out)
scores = pd.DataFrame(rows)

# attach what the app needs to display / search
show_cols = ["unique_id", "name", "facility_type", "operator_type", "address", "city", "state", "pincode",
             "pin_district", "loc_lat", "loc_lon", "loc_source", "flag_location_uncertain",
             "official_phone", "official_website", "n_beds", "n_doctors", "n_flags"]
gold = scores.merge(silver[show_cols], on="unique_id", how="left")
assert not gold.duplicated(["unique_id", "service"]).any()
print(f"{len(gold):,} facility x service rows from {gold['unique_id'].nunique():,} facilities")
summary = (gold.pivot_table(index="service_label", columns="trust_level", values="unique_id",
                            aggfunc="count", fill_value=0)
               .reindex(columns=["Verified", "Likely", "Claimed only"], fill_value=0))
summary["total"] = summary.sum(axis=1)
display(summary.sort_values("total", ascending=False).reset_index())

26,565 facility x service rows from 6,891 facilities


service_label,Verified,Likely,Claimed only,total
Emergency C-section,166,1351,3544,5061
Trauma / emergency care,712,1911,982,3605
CT scan,715,544,2335,3594
MRI scan,447,336,2653,3436
Intensive care (ICU),815,1329,841,2985
Cancer treatment (chemo / radiation),244,1058,1249,2551
Dialysis,314,560,1517,2391
Newborn intensive care (NICU),214,443,922,1579
Cardiac surgery,155,345,629,1129
Burn care,29,105,100,234


In [0]:
# Cell 4: save the Gold table (+ lineage and integrity checks)
import datetime
GOLD = "workspace.default.gold_facility_services"
gold["silver_source"] = "workspace.default.silver_facilities"
gold["silver_version"] = int(spark.sql("DESCRIBE HISTORY workspace.default.silver_facilities LIMIT 1").first()["version"])
gold["gold_built_at"] = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

assert gold["trust_score"].between(0, 100).all()
assert set(gold["trust_level"]) <= {"Verified", "Likely", "Claimed only"}
assert set(gold["unique_id"]) <= set(silver["unique_id"])
assert gold["evidence"].str.len().min() >= 1, "every row must quote at least one piece of evidence"

(spark.createDataFrame(gold)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(GOLD))
spark.sql(f"""COMMENT ON TABLE {GOLD} IS 'Gold: one row per facility x service with any evidence. trust_score 0-100, trust_level Verified/Likely/Claimed only, evidence = quoted source text, score_reasons/problems = how the score was built. Built by 04_gold_scoring from silver_facilities.'""")
saved = spark.table(GOLD)
print(f"saved {GOLD}: {saved.count():,} rows x {len(saved.columns)} columns")

saved workspace.default.gold_facility_services: 26,565 rows x 36 columns


In [0]:
# Cell 5: spot-check samples for human review (2 per service x trust level)
# Read each one and ask: does the quoted evidence really support the level it got?
for key, s in SERVICES.items():
    print("=" * 100)
    print(s["label"].upper())
    for level in ["Verified", "Likely", "Claimed only"]:
        sub = gold[(gold["service"] == key) & (gold["trust_level"] == level)]
        for _, r in sub.sample(min(2, len(sub)), random_state=7).iterrows():
            print(f"\n  [{level} | score {r['trust_score']}] {r['name']} ({r['facility_type']}, {r['city']}, {r['state']})")
            for e in r["evidence"][:2]:
                print(f"      evidence  {e[:160]}")
            print(f"      score     {'; '.join(r['score_reasons'])}")
            if r["problems"]:
                print(f"      problems  {'; '.join(r['problems'])}")

DIALYSIS

  [Verified | score 65] Molvi Ibrahim Mulla Memorial Hospital, 228, Bardoli Road, Kabilpore, Navsari - 396424 (hospital, Kabilpore, Gujarat)
      evidence  procedure/capability: Dialysis (hemodialysis) is provided; four dialysis machines in the department; free dialysis for zakat-eligible patients.
      evidence  procedure/capability: Provides multidisciplinary care across Pediatrics, Gynecology, Orthopedics, ENT, Urology, Dermatology, Dentistry, Ophthalmology, Physiothe
      score     +30 procedure/capability mentions it (3); +25 equipment listed (1); +10 listed as a specialty

  [Verified | score 60] Neelam Hospital (hospital, Rajpura, Punjab)
      evidence  procedure/capability: Provides hemodialysis sessions in sterile, advanced dialysis units
      evidence  procedure/capability: Offers peritoneal dialysis with patient and caregiver training and support
      score     +30 procedure/capability mentions it (7); +15 description mentions it; +10 listed as a specialty; +

In [0]:
# Cell 6: location lookup (PIN / district / city -> coordinates) + recommend() demo
LOOKUP = "workspace.default.gold_location_lookup"
pin_raw = spark.sql("""
  SELECT regexp_extract(CAST(pincode AS STRING), '([0-9]{6})', 1) AS pincode,
         initcap(district) AS district, initcap(statename) AS state, initcap(officename) AS office,
         try_cast(latitude AS DOUBLE) AS lat, try_cast(longitude AS DOUBLE) AS lon
  FROM workspace.default.bronze_india_post_pincode_directory VERSION AS OF 0""").toPandas()
pin_raw = pin_raw[pin_raw["lat"].between(6, 37.5) & pin_raw["lon"].between(68, 97.5)]
key = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())

def centres(df, col, place_type, source):
    # vectorised aggregation (fast); state = first value in the group
    g = (df.dropna(subset=[col]).groupby(col)
           .agg(state=("state", "first"), lat=("lat", "median"), lon=("lon", "median"), n_points=("lat", "size"))
           .reset_index().rename(columns={col: "place_name"}))
    g["place_type"], g["source"] = place_type, source
    return g

# select first, then rename (silver already has lat/lon columns, renaming loc_lat -> lat would duplicate them)
city_pts = silver[["city", "state", "loc_lat", "loc_lon"]].rename(columns={"loc_lat": "lat", "loc_lon": "lon"}).dropna()
lookup = pd.concat([
    centres(pin_raw, "pincode", "pin", "India Post PIN directory"),
    centres(pin_raw, "district", "district", "India Post PIN directory"),
    centres(city_pts, "city", "city", "median of facilities in that city"),
    centres(pin_raw, "office", "post_office", "India Post PIN directory"),
], ignore_index=True)
lookup["place_key"] = lookup["place_name"].map(key)
spark.createDataFrame(lookup).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOOKUP)
print(f"saved {LOOKUP}: {len(lookup):,} places", lookup["place_type"].value_counts().to_dict())

def km(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = (np.radians(x) for x in (la1, lo1, la2, lo2))
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def locate(text):
    """Turn 'Patna' / '800001' / '25.6,85.1' into (lat, lon, description)."""
    t = str(text).strip()
    m = re.fullmatch(r"\s*(-?\d+(\.\d+)?)\s*,\s*(-?\d+(\.\d+)?)\s*", t)
    if m:
        return float(m.group(1)), float(m.group(3)), "coordinates you entered"
    k = key(t)
    order = ["pin", "city", "district", "post_office"] if k.isdigit() else ["city", "district", "post_office", "pin"]
    for ptype in order:
        hit = lookup[(lookup["place_type"] == ptype) & (lookup["place_key"] == k)]
        if len(hit):
            h = hit.sort_values("n_points", ascending=False).iloc[0]
            return h["lat"], h["lon"], f"{ptype.replace('_', ' ')} '{h['place_name']}' ({h['state']}), centre from {h['source']}"
    return None

def recommend(service, location, top_n=5, max_km=200):
    s = SERVICES[service]
    loc = locate(location)
    if loc is None:
        return f"Could not find the place '{location}'. Try a 6-digit PIN code or a nearby city.", None
    lat, lon, how = loc
    c = gold[gold["service"] == service].copy()
    c["distance_km"] = km(lat, lon, c["loc_lat"], c["loc_lon"]).round(1)
    c = c[c["distance_km"] <= max_km]
    # time-critical services weigh distance more: 0.5 points per km vs 0.2
    c["rank_score"] = c["trust_score"] - (0.5 if s["emergency"] else 0.2) * c["distance_km"]
    strong = c[c["trust_level"] != "Claimed only"].sort_values("rank_score", ascending=False)
    picks = strong.head(top_n)
    if len(picks) < 3:   # too few trusted options: show the nearest weak claims too, clearly labelled
        weak = c[c["trust_level"] == "Claimed only"].sort_values("distance_km").head(3 - len(picks))
        picks = pd.concat([picks, weak])
    header = f"{s['label']} near {location}: searched around the {how}; {len(strong)} trusted options within {max_km} km."
    return header, picks

header, picks = recommend("dialysis", "Patna")
print(header)
for i, (_, r) in enumerate(picks.iterrows(), 1):
    approx = " (location approximate)" if r["flag_location_uncertain"] else ""
    print(f"\n{i}. {r['name']} | {r['trust_level']} {r['trust_score']}/100 | {r['distance_km']} km{approx}")
    print(f"   {r['address'] or ''}, {r['city']} | phone: {r['official_phone'] or 'n/a'}")
    for e in r["evidence"][:2]:
        print(f"   evidence: {e[:170]}")
    if r["problems"]:
        print(f"   caution: {'; '.join(r['problems'])}")

saved workspace.default.gold_location_lookup: 154,927 places {'post_office': 133006, 'pin': 19550, 'city': 1621, 'district': 750}
Dialysis near Patna: searched around the city 'Patna' (Bihar), centre from median of facilities in that city; 38 trusted options within 200 km.

1. Hi-Tech Emergency Hospital – Saguna More, Bailey Road, Patna-801503 | Verified 95/100 | 10.3 km
   Saguna More, Bailey Road, Patna | phone: +918102262107
   evidence: description: It has ICU (35 beds), ICCU (15 beds), NICU (15 beds), CT scan, digital X-ray, ultrasound with color Doppler, Echo, dialysis, mammography, radiology, patholog
   evidence: procedure/capability: Advanced Kidney Dialysis

2. Patna Dialysis Centre | Verified 80/100 | 1.0 km
   Shahi Hospital, Road No 2b, Patna | phone: +918541877685
   evidence: procedure/capability: Hemodialysis is performed three or more times a week
   evidence: procedure/capability: Dialysis treatment is conducted by a dialysis technician under the guidance of a nephrol